In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#Ahora implementamos librerias para utilizar los algoritmos de knn, rmse y mse
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial import distance_matrix
from scipy.spatial.distance import euclidean, pdist, squareform
from scipy import sparse
import math  
import sklearn.metrics 
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

## Resumen

1. Crear la matriz de usuarios/películas
2. Solucionar problema escasez de datos
3. Crear la matriz de similitud
4. Visualización de datos con Matplotlib

## 1. Crear la matriz usuarios/películas

In [7]:
df = pd.read_csv("BBDD_100K/ratings.csv")
df.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [17]:
indice=list(df['userId'].unique())
columnas=list(df['movieId'].unique())
indice=sorted(indice)
columnas=sorted(columnas)
 
escasez=pd.pivot_table(data=df,values='rating',index='userId',columns='movieId')
escasez.head(10)

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,4.0,5.0,3.0,5.0,4.0,4.0,3.0,NaN,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,4.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Solucionar problema escasez de datos

Para solucionar la escasez de datos voy a probar normalizando todos los datos de la tabla

1. Primero voy a restar la calificación promedio de cada usuario para centrarla alrededor de 0 \
    2.1 Esto es útil porque facilita las comparaciones entre usuarios y/o ítems, eliminando posibles sesgos debido a diferentes escalas de calificaciones o preferencias personales de los usuarios. 
2. Finalmente voy a convertir los NaN a 0

In [37]:
valoraciones_promedio = escasez.mean(axis = 1)
restar_promedio = escasez.sub(valoraciones_promedio , axis = 0)

In [41]:
restar_promedio = restar_promedio.fillna(0) #Rellenamos valores sobrantes por 0
usuario_pelicula = restar_promedio.copy()

In [42]:
usuario_pelicula.head(10)

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,-0.366379,0.000000,-0.366379,0.000000,0.000000,-0.366379,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.363636,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,0.000000,0.506369,1.506369,-0.493631,1.506369,0.506369,0.506369,-0.493631,0.0,-0.493631,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,1.269737,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,0.000000,0.425532,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,-1.574468,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 2. Crear la matriz de similitud

1. Primero separaré al usuario para predecir sus calificaciones, en nuestro caso lo haremos con el usuario número 3
2. Después eliminaré la película que quiero predecir (en este caso y gracias al EDA utilizaré la película con mas valoraciones, la cual es Forrest Gump, con el id nº356)
3. Finalmente crearé el conjunto de usuarios que vieron esa película en específico 

In [63]:
id_usuario = 3
usuario_objetivo = usuario_pelicula.iloc[[id_usuario]]
print("Usuario Objetivo:")
usuario_objetivo.head()

Usuario Objetivo:


movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [64]:
id_pelicula = 356
pelicula_objetivo = usuario_pelicula.drop( id_pelicula , axis =1)
pelicula_objetivo.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,-0.366379,0.0,-0.366379,0.0,0.0,-0.366379,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.363636,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [87]:
usuarios_que_vieron_la_pelicula_con_nulls = usuario_pelicula[356]  # Selecciono todas las filas respecto a la película en cuestión
usuarios_que_vieron_la_pelicula = usuario_pelicula[usuarios_que_vieron_la_pelicula_con_nulls != 0] # Seleccionamos solo las filas con valores diferentes a 0, de entre los que vieron esa película
usuarios_que_vieron_la_pelicula[356].head(20) #Muestra de los usuarios que vieron la película

userId
1    -0.366379
6     1.506369
7     1.769737
8    -0.574468
10    0.221429
11    1.218750
14    0.604167
15    1.551852
16   -0.224490
17    0.790476
18    0.767928
19   -0.607397
21    1.239278
22    2.428571
24    0.850000
26   -0.238095
27    1.451852
28    0.979825
29    0.358025
33    1.211538
Name: 356, dtype: float64